# Cement marginal abatement cost curve

Create a cement-sector marginal abatement cost curve from the reusable MACC helpers in `src/`. The curve compares each technology with BAU, uses direct stack-emissions abatement only, and excludes carbon-price payments from the cost numerator. The notebook displays outputs inline by default; saving CSV and PNG outputs is optional.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_macc import (
    DEFAULT_MACC_TECHNOLOGIES,
    deterministic_cement_macc,
    generate_cement_macc_outputs,
    plot_cement_macc,
    simulated_cement_macc,
)
from cement.cement_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_RETROFIT_BAU_MODE,
    DEFAULT_SAMPLE_SIZE,
)

pd.options.display.float_format = "{:,.3f}".format

## Settings

Use deterministic mode for a clean representative MACC curve. Use simulated mode to summarize Monte Carlo uncertainty; the headline cost is the aggregate ratio of mean incremental cost to mean abatement, while p05/median/p95 report draw-level cost variation.

In [2]:
USE_SIMULATED = False
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE
TECHNOLOGIES = DEFAULT_MACC_TECHNOLOGIES
SAVE_OUTPUTS = False

mode_label = "simulated" if USE_SIMULATED else "deterministic"
print(f"Selected MACC mode: {mode_label}")
print(f"Technologies: {', '.join(TECHNOLOGIES)}")
if USE_SIMULATED:
    print(f"Monte Carlo draws: {SAMPLE_SIZE:,}")
    print(f"Random seed: {RANDOM_SEED}")
    print(f"Retrofit BAU mode: {RETROFIT_BAU_MODE}")

Selected MACC mode: deterministic
Technologies: electrification, electrolysis, clinker_substitution, alternative_fuels, efficiency_improvement, waste_heat_recovery, ccs, process_heat_integration


## Calculate MACC values

In [3]:
if USE_SIMULATED:
    macc = simulated_cement_macc(
        sample_size=SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
        technologies=TECHNOLOGIES,
        retrofit_bau_mode=RETROFIT_BAU_MODE,
    )
else:
    macc = deterministic_cement_macc(technologies=TECHNOLOGIES)

display(macc)

,technology,label,annual_abatement_tco2,annual_abatement_mtco2,abatement_share_of_bau,incremental_annual_cost_eur,abatement_cost_eur_per_tco2,abatement_cost_p05_eur_per_tco2,abatement_cost_median_eur_per_tco2,abatement_cost_p95_eur_per_tco2
0,efficiency_improvement,Efficiency improvement,"6,000.000",0.006,0.010,"-527,452.093",-87.909,-87.909,-87.909,-87.909
1,process_heat_integration,Process heat integration,"39,000.000",0.039,0.065,"-313,120.047",-8.029,-8.029,-8.029,-8.029
2,alternative_fuels,Alternative fuels,"60,000.000",0.060,0.100,"1,853,986.279",30.900,30.900,30.900,30.900
3,clinker_substitution,Clinker substitution,"75,000.000",0.075,0.125,"3,302,580.000",44.034,44.034,44.034,44.034
4,ccs,CCS,"546,000.000",0.546,0.910,"38,504,268.486",70.521,70.521,70.521,70.521
5,electrification,Electrification,"200,000.000",0.200,0.333,"162,353,766.278",811.769,811.769,811.769,811.769
6,electrolysis,Electrolysis,"500,000.000",0.500,0.833,"461,035,013.368",922.070,922.070,922.070,922.070


## Plot MACC

In [4]:
fig = plot_cement_macc(
    macc,
    title=f"Cement Sector - Marginal Abatement Cost Curve ({mode_label})",
)
plt.show()
plt.close(fig)

/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42563/2019125803.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Optional output files

Set `SAVE_OUTPUTS = True` in the settings cell to write one CSV to `data/processed/` and one PNG to `figures/`.

In [5]:
if SAVE_OUTPUTS:
    saved_paths = generate_cement_macc_outputs(
        project_root=PROJECT_ROOT,
        deterministic=not USE_SIMULATED,
        sample_size=SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    )
    for path in saved_paths:
        print(path.relative_to(PROJECT_ROOT))
else:
    print("SAVE_OUTPUTS is False; no files were written.")

SAVE_OUTPUTS is False; no files were written.
